# Interplanetary Magnetic Field and the Auroral Zones (Dungey, 1961): Implementation / 구현

Dungey의 "열린 자기권" 모델과 Dungey cycle을 시각화하고, 자기 재결합의 물리학을 직접 구현합니다.

We visualize Dungey's "open magnetosphere" model and the Dungey cycle, and implement the physics of magnetic reconnection.

**구현 내용 / Implementation:**
1. Dungey cycle 자기장 위상 시각화 — Fig. 1 확장 재현
2. 남향 vs. 북향 IMF: 자기장 위상 비교
3. 극관 전위와 $S_D$ 전류 패턴 — Fig. 2 재현
4. 재결합률과 극관 전위차 계산
5. IMF $B_z$ 의존성: 현대 우주기상 예보의 핵심
6. 종합 — Dungey cycle의 시간 진화

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Arc
from matplotlib.colors import Normalize
import matplotlib.cm as cm

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Physical constants
R_E = 6.371e6  # Earth radius (m)
mu_0 = 4 * np.pi * 1e-7  # Vacuum permeability (H/m)
B0_eq = 3.1e-5  # Equatorial surface field (T)
m_p = 1.673e-27  # Proton mass (kg)

print("Constants loaded / 상수 로드 완료")

## Part 1: Dungey Cycle 자기장 위상 시각화 — Fig. 1 확장 재현 / Dungey Cycle Field Topology — Extended Fig. 1

Dungey의 Fig. 1은 중성점을 포함하는 평면에서의 자기장선과 플라즈마 흐름을 보여주는 유일한 그림이었습니다. 이를 확장하여 **쌍극자 + 균일 남향 IMF**의 결합 자기장을 직접 계산하고 시각화합니다.

Dungey's Fig. 1 was the only figure showing field lines and plasma flow in the plane containing neutral points. We extend this by directly computing and visualizing the **combined dipole + uniform southward IMF** field.

쌍극자장: $B_r = -2B_0(R_E/r)^3 \cos\theta$, $B_\theta = -B_0(R_E/r)^3 \sin\theta$

균일 남향 IMF: $\mathbf{B}_{IMF} = -B_{IMF}\hat{z}$

In [ ]:
def dipole_field_cartesian(x: np.ndarray, z: np.ndarray, B0: float = 1.0) -> tuple:
    """Compute dipole magnetic field in Cartesian coordinates.

    Dipole moment aligned with +z axis (northward).

    Args:
        x, z: Position arrays (in units of R_E).
        B0: Surface equatorial field strength (normalized).

    Returns:
        Tuple of (Bx, Bz) field components.
    """
    r = np.sqrt(x**2 + z**2)
    r = np.maximum(r, 0.01)  # Avoid singularity
    r5 = r**5
    Bx = 3 * B0 * x * z / r5
    Bz = B0 * (3 * z**2 - r**2) / r5
    return Bx, Bz


def combined_field(x: np.ndarray, z: np.ndarray, B_imf: float = -0.01) -> tuple:
    """Compute combined dipole + uniform IMF field.

    Args:
        x, z: Position arrays (R_E).
        B_imf: IMF Bz component (negative = southward). Normalized to surface B0.

    Returns:
        Tuple of (Bx_total, Bz_total).
    """
    Bx_dip, Bz_dip = dipole_field_cartesian(x, z)
    Bx_total = Bx_dip
    Bz_total = Bz_dip + B_imf
    return Bx_total, Bz_total


def find_neutral_points(B_imf: float = -0.01) -> float:
    """Find dayside neutral point distance for dipole + southward IMF.

    At the subsolar neutral point (x=0, z>0 for northward dipole):
    actually for our geometry, neutral points are on the z-axis.
    On +z axis: Bz = B0/z^3 + B_imf = 0 → z = (-B0/B_imf)^(1/3)

    Args:
        B_imf: Southward IMF (negative value).

    Returns:
        Neutral point distance in R_E.
    """
    return (-1.0 / B_imf) ** (1.0 / 3.0)


# --- Fig. 1 extended reproduction ---
B_imf_south = -0.008  # Southward IMF (normalized)
r_neutral = find_neutral_points(B_imf_south)

fig, ax = plt.subplots(figsize=(14, 10))

# Create field line plot using streamplot
x_range = np.linspace(-15, 15, 400)
z_range = np.linspace(-12, 12, 300)
X, Z = np.meshgrid(x_range, z_range)

Bx, Bz = combined_field(X, Z, B_imf_south)

# Mask inside Earth
R = np.sqrt(X**2 + Z**2)
mask = R < 1.0
Bx[mask] = np.nan
Bz[mask] = np.nan

# Field magnitude for color
B_mag = np.sqrt(Bx**2 + Bz**2)
B_mag[mask] = np.nan

# Streamplot
strm = ax.streamplot(
    X, Z, Bx, Bz,
    color=np.log10(B_mag + 1e-6), cmap="coolwarm",
    density=2.5, linewidth=0.8, arrowsize=1.0,
    minlength=0.3,
)

# Earth
earth = Circle((0, 0), 1.0, facecolor="steelblue", edgecolor="black", lw=2, zorder=10)
ax.add_patch(earth)
ax.text(0, 0, "Earth", ha="center", va="center", fontsize=10,
        fontweight="bold", color="white", zorder=11)

# Mark neutral points
ax.plot(0, r_neutral, "r*", ms=15, zorder=12)
ax.plot(0, -r_neutral, "r*", ms=15, zorder=12)
ax.annotate("Dayside neutral point\n주간측 중성점",
            xy=(0, r_neutral), xytext=(4, r_neutral + 1),
            fontsize=9, color="red",
            arrowprops=dict(arrowstyle="->", color="red"))
ax.annotate("Nightside neutral point\n야간측 중성점",
            xy=(0, -r_neutral), xytext=(4, -r_neutral - 1.5),
            fontsize=9, color="red",
            arrowprops=dict(arrowstyle="->", color="red"))

# Add flow arrows to indicate Dungey cycle
flow_style = dict(arrowstyle="-|>", color="green", lw=2.5)
# Dayside: solar wind brings IMF toward Earth
ax.annotate("", xy=(0, 9), xytext=(0, 12), arrowprops=flow_style)
ax.text(1, 10.5, "Solar wind\n태양풍", fontsize=9, color="green")
# Flanks: open field lines dragged tailward
ax.annotate("", xy=(-8, -3), xytext=(-8, 3), arrowprops=flow_style)
ax.annotate("", xy=(8, -3), xytext=(8, 3), arrowprops=flow_style)
ax.text(-11, 0, "Open lines\ndragged\ntailward\n열린 자기장선\n꼬리 방향", fontsize=8, color="green")
# Nightside: reconnected lines return
ax.annotate("", xy=(0, -6), xytext=(0, -9), arrowprops=flow_style)
ax.text(1, -7.5, "Nightside\nreconnection\n야간측 재결합", fontsize=8, color="green")
# Return flow
ax.annotate("", xy=(-3, 2), xytext=(-3, -2), arrowprops=dict(arrowstyle="-|>", color="orange", lw=2))
ax.annotate("", xy=(3, 2), xytext=(3, -2), arrowprops=dict(arrowstyle="-|>", color="orange", lw=2))
ax.text(3.5, 0, "Return flow\n역방향 흐름", fontsize=8, color="orange")

ax.set_xlabel("X (sunward ←→ tailward) / X (태양 방향 ←→ 꼬리 방향) [$R_E$]")
ax.set_ylabel("Z (southward ↓ ↑ northward) / Z (남 ↓ ↑ 북) [$R_E$]")
ax.set_title("Fig. 1 Extended: Dungey Cycle — Dipole + Southward IMF\n"
             "Fig. 1 확장: Dungey Cycle — 쌍극자 + 남향 IMF", fontsize=13)
ax.set_xlim(-14, 14)
ax.set_ylim(-11, 11)
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

print(f"Neutral point distance: {r_neutral:.2f} R_E ({r_neutral * 6371:.0f} km)")
print(f"IMF strength (normalized): {B_imf_south:.4f} B0")

## Part 2: 남향 vs. 북향 IMF — 자기장 위상 비교 / Southward vs. Northward IMF — Topology Comparison

Dungey의 핵심 주장: IMF 방향에 따라 자기권 위상이 **근본적으로 다르다**. 남향 → 열린 자기권, 북향 → 닫힌 자기권.

Dungey's key claim: magnetosphere topology is **fundamentally different** depending on IMF direction. Southward → open, northward → closed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

x_range = np.linspace(-12, 12, 350)
z_range = np.linspace(-10, 10, 280)
X, Z = np.meshgrid(x_range, z_range)
R = np.sqrt(X**2 + Z**2)
mask = R < 1.0

scenarios = [
    ("Southward IMF (Bz < 0) — OPEN\n남향 IMF — 열린 자기권", -0.008),
    ("Northward IMF (Bz > 0) — CLOSED\n북향 IMF — 닫힌 자기권", +0.008),
]

for ax, (title, b_imf) in zip(axes, scenarios):
    Bx, Bz = combined_field(X, Z, b_imf)
    Bx[mask] = np.nan
    Bz[mask] = np.nan
    B_mag = np.sqrt(Bx**2 + Bz**2)
    B_mag[mask] = np.nan

    ax.streamplot(X, Z, Bx, Bz,
                  color=np.log10(B_mag + 1e-6), cmap="coolwarm",
                  density=2.0, linewidth=0.7, arrowsize=0.8, minlength=0.3)

    earth = Circle((0, 0), 1.0, facecolor="steelblue", edgecolor="black", lw=2, zorder=10)
    ax.add_patch(earth)
    ax.text(0, 0, "E", ha="center", va="center", fontsize=10,
            fontweight="bold", color="white", zorder=11)

    # Mark neutral points if southward
    if b_imf < 0:
        rn = find_neutral_points(b_imf)
        ax.plot(0, rn, "r*", ms=12, zorder=12)
        ax.plot(0, -rn, "r*", ms=12, zorder=12)
        ax.text(0.5, rn + 0.5, "Neutral\npoint", fontsize=8, color="red")
        # Highlight open field line region
        ax.text(6, 0, "OPEN\n열린\n자기장선", fontsize=10, color="red",
                fontweight="bold", ha="center",
                bbox=dict(facecolor="lightyellow", alpha=0.8))
    else:
        ax.text(6, 0, "CLOSED\n닫힌\n자기장선", fontsize=10, color="blue",
                fontweight="bold", ha="center",
                bbox=dict(facecolor="lightcyan", alpha=0.8))

    ax.set_title(title, fontsize=11)
    ax.set_xlim(-11, 11)
    ax.set_ylim(-9, 9)
    ax.set_aspect("equal")
    ax.set_xlabel("X [$R_E$]")
    ax.set_ylabel("Z [$R_E$]")

plt.suptitle("Dungey's Key Insight: IMF Direction Controls Magnetosphere Topology\n"
             "Dungey의 핵심 통찰: IMF 방향이 자기권 위상을 결정", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 3: 극관 전위와 $S_D$ 전류 패턴 — Fig. 2 재현 / Polar Cap Potential and $S_D$ Current Pattern — Fig. 2 Reproduction

Dungey의 Fig. 2는 북반구 전리층의 등전위선을 보여주며, 이것이 $S_D$ 전류 선과 닮는다고 주장했습니다. 극관(polar cap) 내외의 전위 구조를 재현합니다.

Dungey's Fig. 2 showed ionospheric equipotentials in the northern hemisphere, arguing they resemble $S_D$ current lines. We reproduce the potential structure inside and outside the polar cap.

핵심: 곡선 $A$(열린/닫힌 자기장선 경계 = 오로라 타원)에서 **전위가 역전**됩니다.

Key: **Potential reversal** at curve $A$ (open/closed field line boundary = auroral oval).

In [ ]:
def polar_cap_potential(
    colat_deg: np.ndarray,
    mlt_hr: np.ndarray,
    phi_pc: float = 60.0,
    colat_boundary: float = 15.0,
) -> np.ndarray:
    """Compute ionospheric potential for a two-cell Dungey convection pattern.

    Inside polar cap: E points dawn-to-dusk (potential varies with MLT).
    Outside polar cap: Potential reverses (return flow).

    Args:
        colat_deg: Magnetic colatitude (degrees from pole).
        mlt_hr: Magnetic local time (hours, 0=midnight, 12=noon).
        phi_pc: Cross-polar-cap potential drop (kV).
        colat_boundary: Colatitude of open/closed boundary (degrees).

    Returns:
        Potential (kV).
    """
    mlt_rad = mlt_hr * np.pi / 12.0  # Convert to radians (0=midnight)

    # Dawn-dusk direction: sin(MLT) where MLT=6hr is dawn, 18hr is dusk
    dawn_dusk = np.sin(mlt_rad)  # +1 at dawn (6hr), -1 at dusk (18hr)

    # Inside polar cap: linear variation
    inside = colat_deg < colat_boundary
    potential = np.zeros_like(colat_deg, dtype=float)

    # Inside: potential = phi_pc/2 * sin(MLT) * (colat/colat_boundary)
    potential[inside] = (phi_pc / 2) * dawn_dusk[inside] * (colat_deg[inside] / colat_boundary)

    # Outside: potential reversal, decaying with distance from boundary
    outside = ~inside
    decay = np.exp(-(colat_deg[outside] - colat_boundary) / 10.0)
    potential[outside] = -(phi_pc / 2) * dawn_dusk[outside] * decay * 0.5

    return potential


fig, axes = plt.subplots(1, 2, figsize=(16, 7), subplot_kw=dict(polar=True))

# Grid in polar coordinates
colat = np.linspace(0, 50, 200)  # Colatitude (degrees from pole)
mlt = np.linspace(0, 24, 200)   # MLT (hours)
COLAT, MLT = np.meshgrid(colat, mlt)

phi = polar_cap_potential(COLAT, MLT, phi_pc=60.0, colat_boundary=15.0)

# Convert to polar plot coordinates
theta_plot = MLT * np.pi / 12.0  # MLT to radians (0=midnight at bottom)
r_plot = COLAT  # Colatitude as radius

# Left: Equipotentials (Dungey's Fig. 2)
ax = axes[0]
levels = np.linspace(-30, 30, 21)
cs = ax.contour(theta_plot, r_plot, phi, levels=levels, cmap="RdBu_r", linewidths=1.2)
ax.set_theta_zero_location("S")  # Midnight at bottom
ax.set_theta_direction(-1)  # Clockwise (standard MLT)
ax.set_ylim(0, 40)
ax.set_yticks([10, 20, 30, 40])
ax.set_yticklabels(["80°", "70°", "60°", "50°"], fontsize=8)

# Mark auroral oval (curve A)
theta_oval = np.linspace(0, 2 * np.pi, 100)
ax.plot(theta_oval, np.full_like(theta_oval, 15), "k-", lw=2.5,
        label="Curve A (auroral oval)\n곡선 A (오로라 타원)")

# MLT labels
for hr, label in [(0, "00\nMidnight"), (6, "06\nDawn"), (12, "12\nNoon"), (18, "18\nDusk")]:
    ax.text(hr * np.pi / 12, 45, label, ha="center", fontsize=8)

ax.set_title("Fig. 2 Reproduction: Equipotentials\n등전위선 (Dungey Fig. 2)", pad=20)
ax.legend(loc="lower left", fontsize=8, bbox_to_anchor=(-0.1, -0.15))

# Right: Current pattern (Hall current, j ⊥ E)
ax = axes[1]
# Hall current is perpendicular to E, so current lines ≈ equipotentials
# when Hall conductivity dominates
cs2 = ax.contour(theta_plot, r_plot, phi, levels=levels, cmap="RdBu_r", linewidths=1.2)
ax.set_theta_zero_location("S")
ax.set_theta_direction(-1)
ax.set_ylim(0, 40)
ax.set_yticks([10, 20, 30, 40])
ax.set_yticklabels(["80°", "70°", "60°", "50°"], fontsize=8)

ax.plot(theta_oval, np.full_like(theta_oval, 15), "k-", lw=2.5)

# Add current direction arrows
for hr in [3, 9, 15, 21]:
    theta_arr = hr * np.pi / 12
    # Inside polar cap: antisunward (toward midnight)
    if hr in [3, 21]:
        ax.annotate("", xy=(theta_arr, 10), xytext=(theta_arr, 5),
                    arrowprops=dict(arrowstyle="->", color="green", lw=2))
    # Outside: sunward (toward noon)
    if hr in [9, 15]:
        ax.annotate("", xy=(theta_arr, 25), xytext=(theta_arr, 20),
                    arrowprops=dict(arrowstyle="->", color="orange", lw=2))

for hr, label in [(0, "00"), (6, "06"), (12, "12"), (18, "18")]:
    ax.text(hr * np.pi / 12, 45, label, ha="center", fontsize=8)

ax.set_title("$S_D$ Current Pattern (Hall current ⊥ E)\n$S_D$ 전류 패턴 (Hall 전류 ⊥ E)", pad=20)

# Add text explaining the key insight
fig.text(0.5, -0.02,
         "Key: Equipotentials ≈ Current lines because Hall conductivity dominates (σ_H >> σ_P)\n"
         "핵심: Hall 전도도가 지배적(σ_H >> σ_P)이므로 등전위선 ≈ 전류선",
         ha="center", fontsize=10, style="italic")

plt.tight_layout()
plt.show()

## Part 4: 재결합률과 극관 전위차 / Reconnection Rate and Cross-Polar-Cap Potential

$\Phi_{PC} = v_{SW} \cdot |B_z| \cdot L_{\text{eff}}$

태양풍 속도와 IMF $B_z$에 따른 극관 전위차를 계산합니다. 이 전위차가 Dungey cycle의 강도를 결정합니다.

Computing cross-polar-cap potential as a function of solar wind speed and IMF $B_z$. This potential determines the Dungey cycle intensity.

In [ ]:
def cross_polar_cap_potential(
    v_sw_km_s: float,
    bz_nT: float,
    L_eff_RE: float = 7.0,
) -> float:
    """Compute cross-polar-cap potential from Dungey's model.

    Args:
        v_sw_km_s: Solar wind speed (km/s).
        bz_nT: IMF Bz component (nT, negative = southward).
        L_eff_RE: Effective reconnection length (Earth radii).

    Returns:
        Cross-polar-cap potential (kV).
    """
    v_sw = v_sw_km_s * 1e3  # m/s
    bz = abs(bz_nT) * 1e-9  # T (use absolute value for southward)
    L_eff = L_eff_RE * R_E  # m

    # E_y = v_sw * |Bz| (dawn-dusk electric field)
    E_y = v_sw * bz  # V/m

    # Phi_PC = E_y * L_eff
    phi_kV = E_y * L_eff / 1e3  # kV

    return phi_kV


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Phi_PC vs Bz for different solar wind speeds
ax = axes[0]
bz_range = np.linspace(-20, 0, 100)  # Southward only
for v_sw in [300, 400, 500, 600, 800]:
    phi = [cross_polar_cap_potential(v_sw, bz) for bz in bz_range]
    ax.plot(-bz_range, phi, lw=2, label=f"$v_{{SW}}$ = {v_sw} km/s")

ax.axhspan(30, 100, color="yellow", alpha=0.1)
ax.text(15, 85, "Typical range\n전형적 범위\n30–100 kV", fontsize=8, color="orange")
ax.set_xlabel("|IMF $B_z$| (nT, southward / 남향)")
ax.set_ylabel("$\\Phi_{PC}$ (kV)")
ax.set_title("(a) Polar Cap Potential vs. IMF $B_z$\n극관 전위차 vs. IMF $B_z$")
ax.legend(fontsize=8)
ax.set_xlim(0, 20)
ax.set_ylim(0, 150)

# (b) Phi_PC vs V_sw for different Bz
ax = axes[1]
v_range = np.linspace(200, 1000, 100)
for bz in [-2, -5, -10, -15, -20]:
    phi = [cross_polar_cap_potential(v, bz) for v in v_range]
    ax.plot(v_range, phi, lw=2, label=f"$B_z$ = {bz} nT")

ax.axhspan(30, 100, color="yellow", alpha=0.1)
ax.set_xlabel("Solar wind speed / 태양풍 속도 (km/s)")
ax.set_ylabel("$\\Phi_{PC}$ (kV)")
ax.set_title("(b) Polar Cap Potential vs. $v_{SW}$\n극관 전위차 vs. 태양풍 속도")
ax.legend(fontsize=8)

# (c) 2D heatmap: Phi_PC(v_sw, Bz)
ax = axes[2]
v_grid = np.linspace(200, 900, 80)
bz_grid = np.linspace(-25, -1, 80)
V, BZ = np.meshgrid(v_grid, bz_grid)
PHI = np.vectorize(cross_polar_cap_potential)(V, BZ)

im = ax.pcolormesh(v_grid, -bz_grid, PHI, cmap="hot_r", shading="auto")
plt.colorbar(im, ax=ax, label="$\\Phi_{PC}$ (kV)")

# Contour lines
cs = ax.contour(v_grid, -bz_grid, PHI, levels=[30, 60, 100, 150],
                colors="white", linewidths=1)
ax.clabel(cs, fmt="%.0f kV", fontsize=8, colors="white")

# Mark typical conditions
ax.plot(400, 5, "c*", ms=12)
ax.text(420, 6, "Quiet\n조용", fontsize=8, color="cyan")
ax.plot(600, 15, "y*", ms=12)
ax.text(620, 16, "Storm\n폭풍", fontsize=8, color="yellow")

ax.set_xlabel("$v_{SW}$ (km/s)")
ax.set_ylabel("|$B_z$| (nT, southward / 남향)")
ax.set_title("(c) $\\Phi_{PC}$ Heatmap\n극관 전위차 히트맵")

plt.tight_layout()
plt.show()

# Print key values
print("Cross-polar-cap potential examples:")
print(f"  Quiet (400 km/s, Bz=-5 nT): {cross_polar_cap_potential(400, -5):.1f} kV")
print(f"  Moderate (500 km/s, Bz=-10 nT): {cross_polar_cap_potential(500, -10):.1f} kV")
print(f"  Storm (700 km/s, Bz=-20 nT): {cross_polar_cap_potential(700, -20):.1f} kV")

## Part 5: IMF $B_z$ 의존성 — 현대 우주기상 예보의 핵심 / IMF $B_z$ Dependence — Core of Modern Space Weather Forecasting

Dungey의 모델이 현대 우주기상 예보의 기초가 된 이유: **IMF $B_z$가 자기권 활동의 가장 중요한 매개변수**입니다. Burton et al. (1975, 논문 #11)의 경험적 Dst 공식은 Dungey의 이론적 예측을 정량적으로 확인했습니다.

Why Dungey's model became the foundation of modern forecasting: **IMF $B_z$ is the most important parameter for magnetospheric activity**. Burton et al.'s (1975, paper #11) empirical Dst formula quantitatively confirmed Dungey's theoretical prediction.

In [ ]:
def burton_dst(
    bz_nT: np.ndarray,
    v_sw: float = 400.0,
    n_sw: float = 5.0,
    tau: float = 7.7,
    dt_hr: float = 0.1,
    duration_hr: float = 24.0,
) -> tuple:
    """Simplified Burton et al. (1975) Dst prediction.

    dDst/dt = Q(E_y) - Dst/tau

    where Q is the energy injection function depending on dawn-dusk E field.

    Args:
        bz_nT: Time series of IMF Bz (nT, negative = southward).
        v_sw: Solar wind speed (km/s), constant.
        n_sw: Solar wind density (cm^-3), constant.
        tau: Ring current decay time (hours).
        dt_hr: Time step (hours).
        duration_hr: Total duration (hours).

    Returns:
        Tuple of (time_hr, Dst_nT).
    """
    n_steps = int(duration_hr / dt_hr)
    t = np.linspace(0, duration_hr, n_steps)

    # Interpolate Bz to time grid
    bz_interp = np.interp(t, np.linspace(0, duration_hr, len(bz_nT)), bz_nT)

    # Dawn-dusk electric field E_y = v_sw * |Bs| (only southward component)
    Bs = np.where(bz_interp < 0, -bz_interp, 0)  # Southward component (positive)
    E_y = v_sw * Bs * 1e-3  # mV/m

    # Dynamic pressure correction
    p_dyn = n_sw * (v_sw * 1e3) ** 2 * m_p * 1e6  # Pa
    b_correction = 7.26 * np.sqrt(p_dyn * 1e9)  # nT (approximate)

    Dst = np.zeros(n_steps)
    for i in range(1, n_steps):
        # Injection: Q = -4.4 * (E_y - 0.5) for E_y > 0.5 mV/m
        if E_y[i] > 0.5:
            Q = -4.4 * (E_y[i] - 0.5)  # nT/hr
        else:
            Q = 0
        # Decay
        dDst = (Q - Dst[i - 1] / tau) * dt_hr
        Dst[i] = Dst[i - 1] + dDst

    return t, Dst


# Simulate three scenarios
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

duration = 48.0  # hours
t_base = np.linspace(0, duration, 500)

scenarios = [
    ("Quiet: Bz ~ +5 nT (northward)\n조용: Bz ~ +5 nT (북향)",
     5 + 2 * np.sin(2 * np.pi * t_base / 6), "green"),
    ("Moderate storm: Bz turns southward (-10 nT) for 6 hours\n중간 폭풍: Bz가 6시간 남향 (-10 nT)",
     np.where((t_base > 12) & (t_base < 18), -10, 3), "orange"),
    ("Intense storm: Bz = -20 nT sustained for 12 hours\n강한 폭풍: Bz = -20 nT 12시간 지속",
     np.where((t_base > 6) & (t_base < 18), -20, 2), "red"),
]

for ax_idx, (label, bz_profile, color) in enumerate(scenarios):
    t_dst, dst = burton_dst(bz_profile, duration_hr=duration)

    # Top subplot: Bz
    ax = axes[0]
    ax.plot(t_base, bz_profile, color=color, lw=2, label=label.split("\n")[0])
    ax.axhline(0, color="gray", ls="--", alpha=0.3)

axes[0].fill_between([0, duration], [-25, -25], [0, 0], color="red", alpha=0.05)
axes[0].text(duration - 5, -22, "Southward\n남향", fontsize=8, color="red")
axes[0].fill_between([0, duration], [0, 0], [10, 10], color="blue", alpha=0.05)
axes[0].text(duration - 5, 5, "Northward\n북향", fontsize=8, color="blue")
axes[0].set_ylabel("IMF $B_z$ (nT)")
axes[0].set_title("IMF $B_z$ Dependence of Geomagnetic Activity (Dungey → Burton)\n"
                   "지자기 활동의 IMF $B_z$ 의존성 (Dungey → Burton)")
axes[0].legend(fontsize=8, loc="lower left")
axes[0].set_ylim(-25, 10)

# Middle: Phi_PC
for label, bz_profile, color in scenarios:
    phi_pc = np.array([cross_polar_cap_potential(400, bz) if bz < 0 else 0
                       for bz in bz_profile])
    axes[1].plot(t_base, phi_pc, color=color, lw=2)

axes[1].set_ylabel("$\\Phi_{PC}$ (kV)")
axes[1].axhline(60, color="gray", ls=":", alpha=0.5)
axes[1].text(1, 65, "Moderate activity threshold", fontsize=8, color="gray")

# Bottom: Dst
for label, bz_profile, color in scenarios:
    t_dst, dst = burton_dst(bz_profile, duration_hr=duration)
    axes[2].plot(t_dst, dst, color=color, lw=2)

axes[2].set_ylabel("Dst (nT)")
axes[2].set_xlabel("Time / 시간 (hours)")
axes[2].axhline(-50, color="orange", ls=":", alpha=0.5)
axes[2].text(1, -45, "Moderate storm (Dst < -50 nT)", fontsize=8, color="orange")
axes[2].axhline(-100, color="red", ls=":", alpha=0.5)
axes[2].text(1, -95, "Intense storm (Dst < -100 nT)", fontsize=8, color="red")

plt.tight_layout()
plt.show()

print("Dungey's prediction confirmed by Burton et al. (1975):")
print("  Southward IMF → Energy injection → Dst decrease (storm)")
print("  Northward IMF → No injection → Quiet conditions")

## Summary / 요약

| Part | 구현 내용 / Implementation | Dungey (1961)의 기여 / Contribution |
|------|--------------------------|--------------------------------------|
| 1 | Dungey cycle 자기장 위상 (Fig. 1 확장) | 쌍극자 + 남향 IMF → 중성점과 열린 자기장선 / Dipole + southward IMF → neutral points and open field lines |
| 2 | 남향 vs. 북향 IMF 위상 비교 | 핵심 통찰: IMF 방향이 자기권 위상을 결정 / Key insight: IMF direction controls topology |
| 3 | 극관 전위와 $S_D$ 전류 (Fig. 2) | 위상에서 관측 가능한 전류 체계 도출 / Observable current system derived from topology |
| 4 | 극관 전위차 계산 | $\Phi_{PC} = v_{SW} \cdot |B_z| \cdot L_{\text{eff}}$ — Dungey cycle 강도 정량화 / Quantifying Dungey cycle intensity |
| 5 | IMF $B_z$ → Dst 의존성 | Burton (1975) 공식으로 Dungey 예측의 정량적 확인 / Quantitative confirmation via Burton equation |

**다음 논문 / Next paper**: #7 Axford & Hines (1961) — 점성 상호작용 모델 — Dungey의 재결합과 함께 자기권 물리학의 2대 패러다임 / Viscous interaction model — together with Dungey's reconnection, the two fundamental paradigms of magnetospheric physics.